In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from kuramoto.config import (
    SimulationConfig,
    GridConfig,
    CouplingConfig,
    InitThetaConfig,
    InitOmegaConfig,
    KernelComponentConfig,
    build_simulation,
)
from kuramoto.analysis import order_parameter
from kuramoto.adjoint import grads_final_R, grads_mean_R
from kuramoto.plotting import plot_2d
from kuramoto.network import (
    create_cortical_graph,
    plot_cortical_graph,
    get_degree,
    get_deg_centrality,
    get_closeness_centrality,
    get_betweenness_centrality,
    get_eigenvector_centrality,
)

SEED = 42
grid_shape = (12, 12)

# Bottleneck Example case
comps_bottleneck = [
KernelComponentConfig(
    kernel="gaussian",
    base_strength=1.0,
    kernel_params={"sigma": 1.0},
    radius=2.0,
    node_groups=[0],
    edge_mode="within",
),
KernelComponentConfig(
    kernel="gaussian",
    base_strength=1.0,
    kernel_params={"sigma": 1.0},
    radius=2.0,
    node_groups=[1],
    edge_mode="within",
),
# bottleneck-like hub (group 2) coupling to/from everyone
KernelComponentConfig(
    kernel="gaussian",
    base_strength=0.8,
    kernel_params={"sigma": 4.0},
    radius=4.0,
    node_groups=[2],
    edge_mode="outgoing",
),
KernelComponentConfig(
    kernel="gaussian",
    base_strength=0.8,
    kernel_params={"sigma": 4.0},
    radius=4.0,
    node_groups=[2],
    edge_mode="incoming",
),
# weak one-way coupling from group 1 to group 0
KernelComponentConfig(
    kernel="gaussian",
    base_strength=0.2,
    kernel_params={"sigma": 2.0},
    radius=2.0,
    node_groups=[1],
    edge_mode="custom",
    to_node_groups=[0],
),
# Apply dropout to all groups
KernelComponentConfig(
kernel="dropout",
kernel_params={"dropout_frac": 0.3},
node_groups=[0,1,2],
edge_mode="within",
seed=SEED,
)
]

# unstructured network example
# Size factors
N_GROUPS = 6
SIZE_LB = 0.5
SIZE_UB = 2.0
size_factors = np.random.uniform(SIZE_LB, SIZE_UB, size=N_GROUPS-1)

comps: list[KernelComponentConfig] = []
for g, sf in enumerate(size_factors):
    comps.append(
        KernelComponentConfig(
            kernel="gaussian",
            base_strength=1.0,
            kernel_params={"sigma": 1.0*sf},
            radius=3.0*sf,
            node_groups=[g],
            edge_mode="within",
        )
    )

# Only one mexican hat component
mh_params = {"sigma_e": 1.0, "sigma_i": 3.0, "a_e": 1.0, "a_i": 0.8}
comps.append(
    KernelComponentConfig(
        kernel="mexican_hat",
        base_strength=0.3,
        kernel_params=mh_params,
        radius=4.0,
        node_groups=[N_GROUPS-1],
        edge_mode="within",
    )
)

# Apply dropout to all groups
comps.append(KernelComponentConfig(
    kernel="dropout",
    kernel_params={"dropout_frac": 0.3},
    node_groups=list(range(n_groups)),
    edge_mode="within",
    seed=seed,
))

sim_simple = build_simulation(config=cfg, rng=np.random.default_rng(SEED))

In [ ]:
G = create_cortical_graph(sim_simple)

plot_cortical_graph(G, layout="grid")

# Run networks for various k to find suitable k for analyses
res_bottleneck = sim_bottleneck.run((0, T_END), dt)
res_unstructured = sim_unstructured.run((0, T_END), dt)

R_bottleneck, _ = order_parameter(res_bottleneck["theta"])
R_unstructured, _ = order_parameter(res_unstructured["theta"])
R_m_bottleneck = np.mean(R_bottleneck, axis=1)
R_m_unstructured = np.mean(R_unstructured, axis=1)
R_link_bottleneck = R_link(res_bottleneck['theta'])
R_link_unstructured = R_link(res_unstructured['theta'])

fig, axs = plt.subplots(2,2,figsize=(12,5), sharey=True)
axs=axs.flatten()
axs[0].plot(res_bottleneck['ts'],R_bottleneck,label='Bottleneck')
axs[1].plot(res_bottleneck['ts'],R_unstructured,label='Unstructured')
axs[0].plot(res_bottleneck['ts'],R_link_bottleneck,label='Bottleneck')
axs[1].plot(res_bottleneck['ts'],R_link_unstructured,label='Unstructured')  
axs[0].legend()
axs[1].legend()
axs[0].set_xlabel('Time')
axs[1].set_xlabel('Time')
axs[0].set_ylabel('Order parameter')
axs[0].set_title('Bottleneck')
axs[1].set_title('Unstructured')
plt.tight_layout()
plt.show()


In [ ]:
deg = get_degree(G)
deg_cent = get_deg_centrality(G)
closeness = get_closeness_centrality(G)
betweenness = get_betweenness_centrality(G)
eigenvector = get_eigenvector_centrality(G)

fig,ax = plt.subplots(1,5,figsize=(12,16),constrained_layout=True)
im = ax[0].imshow(deg.reshape(grid_shape))
ax[0].set_title("Degree")
fig.colorbar(im,ax=ax[0],fraction=0.046, pad=0.04)
im = ax[1].imshow(deg_cent.reshape(grid_shape))
ax[1].set_title("Degree Centrality")
fig.colorbar(im,ax=ax[1],fraction=0.046, pad=0.04)
im = ax[2].imshow(closeness.reshape(grid_shape))
ax[2].set_title("Closeness Centrality")
fig.colorbar(im,ax=ax[2],fraction=0.046, pad=0.04)
im = ax[3].imshow(betweenness.reshape(grid_shape))
ax[3].set_title("Betweenness Centrality")
fig.colorbar(im,ax=ax[3],fraction=0.046, pad=0.04)
im = ax[4].imshow(eigenvector.reshape(grid_shape))
ax[4].set_title("Eigenvector Centrality")
fig.colorbar(im,ax=ax[4],fraction=0.046, pad=0.04)

### 2) Heterogeneous sim

In [ ]:
# Heterogeneous graph
n_rows, n_cols = grid_shape
group_ids = np.zeros((n_rows, n_cols), dtype=int)
group_ids[n_rows // 2 :, :] = 1 # Top half is group 1, bottom half is group 0
group_ids = group_ids.ravel().tolist()

components = [
    KernelComponentConfig(
        kernel="gaussian",
        base_strength=1.0,
        kernel_params={"sigma": 1.0},
        radius=2.0,
        node_groups=[0],
        edge_mode="within",
    ),
    KernelComponentConfig(
        kernel="gaussian",
        base_strength=0.8,
        kernel_params={"sigma": 3.0},
        radius=4.0,
        node_groups=[1],
        edge_mode="within",
    ),
    KernelComponentConfig(
        kernel="gaussian",
        base_strength=0.4,
        kernel_params={"sigma": 2.0},
        radius=4.0,
        node_groups=[1],
        edge_mode="custom",
        to_node_groups=[0],
    ),
    # KernelComponentConfig(
    #     kernel="gaussian",
    #     base_strength=0.4,
    #     kernel_params={"sigma": 2.0},
    #     radius=4.0,
    #     node_groups=[0],
    #     edge_mode="custom",
    #     to_node_groups=[1],
    # ),
]

cfg = SimulationConfig(
    grid=GridConfig(shape=grid_shape, periodic=False),
    coupling=CouplingConfig(
        kernel="gaussian",
        base_strength=1.0,
        radius=4.0,
        mode="spatial",
        components=components,
        group_ids=group_ids,
    ),
    initial_theta=InitThetaConfig(mode="uniform"),
    initial_omega=InitOmegaConfig(mode="normal", mu=0.0, sigma=0.3),
    seed=42,
)

sim_hetero = build_simulation(config=cfg, rng=np.random.default_rng(SEED))

G = create_cortical_graph(sim_hetero)

plot_cortical_graph(G, layout="grid")

In [ ]:
from kuramoto.plotting import plot_coupling_matrix
plot_coupling_matrix(sim_hetero.coupling.K)

In [ ]:
deg = get_degree(G)
deg_cent = get_deg_centrality(G)
closeness = get_closeness_centrality(G)
betweenness = get_betweenness_centrality(G)
eigenvector = get_eigenvector_centrality(G)

fig,ax = plt.subplots(1,5,figsize=(12,16),constrained_layout=True)
im = ax[0].imshow(deg.reshape(grid_shape))
ax[0].set_title("Degree")
fig.colorbar(im,ax=ax[0],fraction=0.046, pad=0.04)
im = ax[1].imshow(deg_cent.reshape(grid_shape))
ax[1].set_title("Degree Centrality")
fig.colorbar(im,ax=ax[1],fraction=0.046, pad=0.04)
im = ax[2].imshow(closeness.reshape(grid_shape))
ax[2].set_title("Closeness Centrality")
fig.colorbar(im,ax=ax[2],fraction=0.046, pad=0.04)
im = ax[3].imshow(betweenness.reshape(grid_shape))
ax[3].set_title("Betweenness Centrality")
fig.colorbar(im,ax=ax[3],fraction=0.046, pad=0.04)
im = ax[4].imshow(eigenvector.reshape(grid_shape))
ax[4].set_title("Eigenvector Centrality")
fig.colorbar(im,ax=ax[4],fraction=0.046, pad=0.04)

In [ ]:
T_END = 5.0
dt = 0.1
RNG = np.random.default_rng(42)

results = sim_hetero.run((0, T_END), dt, rng=RNG)
t_list = results['ts'].tolist()
state_list = [{'theta': th, 'theta_dot': dth, 'omega': results['omega']} for th, dth in zip(results['theta'], results['theta_dot'])]

# Postprocess
R_list, _ = order_parameter(results['theta'])

In [ ]:
fig,ax = plt.subplots(figsize=(10,5))
ax.plot(results["ts"], R_list, linewidth=1.5)